# SC4020 Project 2 — Task 2: Recommending Targets for a Ligand (Collaborative Filtering)

**Users = ligands, items = targets.** The question is *"for this ligand, which targets will it bind most strongly?"*, so each ligand plays the role of a user and its binding scores play the role of ratings on targets (items).

This notebook needs `data/clean/` from `task1_analysis.ipynb`. Scores: DAVIS $\mathrm{p}K_d$, KIBA supplied scores (higher = stronger for both).

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import kendalltau

SEED = 42                      # shared group seed
CLEAN_DIR = Path("data/clean")
MODEL_DIR = Path("saved_models")
FIG_DIR = Path("figures")
MODEL_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

SCALES = {
    "davis": {"score_name": "pKd", "units": "pKd = 9 - log10(Kd [nM])", "screening_limit": 5.0},
    "kiba":  {"score_name": "KIBA score", "units": "supplied KIBA scale (higher = stronger)", "screening_limit": np.nan},
}
pairs = {name: pd.read_csv(CLEAN_DIR / f"{name}_pairs.csv", dtype={"ligand_key": str, "target_key": str})
         for name in SCALES}
{name: len(df) for name, df in pairs.items()}

## Step 1: Train / validation / test split

Pairs are shuffled with a fixed seed and split 70 / 15 / 15. Each pair lands in exactly one partition. CF cannot predict for a ligand or target it has never seen, so any validation/test pair whose ligand or target is missing from training is moved into training (repeated until no such pair remains).

In [ ]:
def split_pairs(df, seed=SEED, fracs=(0.70, 0.15, 0.15)):
    rng = np.random.default_rng(seed)
    order = rng.permutation(len(df))
    n_tr, n_va = int(fracs[0] * len(df)), int(fracs[1] * len(df))
    split = np.empty(len(df), dtype=object)
    split[order[:n_tr]] = "train"
    split[order[n_tr:n_tr + n_va]] = "val"
    split[order[n_tr + n_va:]] = "test"
    df = df.assign(split=split)

    moved = 0
    while True:
        train = df[df["split"] == "train"]
        unseen = (df["split"] != "train") & (~df["ligand_key"].isin(train["ligand_key"]) |
                                             ~df["target_key"].isin(train["target_key"]))
        if not unseen.any():
            break
        moved += int(unseen.sum())
        df.loc[unseen, "split"] = "train"
    return df, moved

splits, split_rows = {}, []
for name, df in pairs.items():
    splits[name], moved = split_pairs(df)
    counts = splits[name]["split"].value_counts()
    split_rows.append({"dataset": name.upper(), **{s: counts[s] for s in ["train", "val", "test"]},
                       "moved to train (unseen ligand/target)": moved})

# checks required by the brief
for name, df in splits.items():
    assert not df.duplicated(["ligand_key", "target_key"]).any()          # each pair in one partition only
    tr = df[df["split"] == "train"]
    ev = df[df["split"] != "train"]
    assert ev["ligand_key"].isin(tr["ligand_key"]).all() and ev["target_key"].isin(tr["target_key"]).all()

split_table = pd.DataFrame(split_rows).set_index("dataset")
split_table["train %"] = (100 * split_table["train"] / split_table[["train", "val", "test"]].sum(axis=1)).round(1)
split_table

## Step 2: The models (lecture 11.3)

**Baseline predictor (global effects, slides 18–22):**

$$b_{ui} = \mu + b_u + b_i, \qquad b_u = \bar r_u - \mu, \qquad b_i = \bar r_i - \mu$$

$\mu$ = mean training score, $\bar r_u$ = mean training score of ligand $u$, $\bar r_i$ = mean training score of target $i$.

**Similarity: centred cosine (slide 9).** Subtract each target's mean from its scores, treat unmeasured entries as 0, then take the cosine between two targets' score vectors:

$$s_{ij} = \frac{\sum_u (r_{ui} - \bar r_i)(r_{uj} - \bar r_j)}{\sqrt{\sum_u (r_{ui} - \bar r_i)^2}\,\sqrt{\sum_u (r_{uj} - \bar r_j)^2}}$$

**Prediction: baseline + neighbour deviations (slides 23, 24, 28).** Item-based (target–target) version:

$$\hat r_{ui} = b_{ui} + \frac{\sum_{j \in N_k(i;u)} s_{ij}\,(r_{uj} - b_{uj})}{\sum_{j \in N_k(i;u)} s_{ij}}$$

$N_k(i;u)$ = the $k$ targets most similar to $i$ that ligand $u$ has a training score for. User-based (ligand–ligand) CF is the same with the roles of ligands and targets swapped.

Only neighbours with positive similarity are used: with negative weights the denominator $\sum s_{ij}$ can approach zero and blow up the prediction.

**Fallback:** if a pair has no positive-similarity neighbour, the prediction is the baseline $b_{ui}$.

In [ ]:
def build_index(df):
    ligands = np.array(sorted(df["ligand_key"].unique()))
    targets = np.array(sorted(df["target_key"].unique()))
    return ligands, targets

def to_idx(keys, vocab):
    pos = {k: i for i, k in enumerate(vocab)}
    return np.array([pos[k] for k in keys])

def fit_baseline(l, t, y, n_l, n_t):
    # slides 18-22: b_u = mean of ligand u - mu, b_i = mean of target i - mu
    mu = y.mean()
    b_l = np.bincount(l, y, n_l) / np.maximum(np.bincount(l, minlength=n_l), 1) - mu
    b_t = np.bincount(t, y, n_t) / np.maximum(np.bincount(t, minlength=n_t), 1) - mu
    return mu, b_l, b_t

def centred_cosine(R):
    # slide 9: subtract each row's mean over its measured entries, unmeasured become 0, then ordinary cosine
    C = np.nan_to_num(R - np.nanmean(R, axis=1, keepdims=True))
    norms = np.linalg.norm(C, axis=1)
    with np.errstate(invalid="ignore", divide="ignore"):
        S = np.where(np.outer(norms, norms) > 0, (C @ C.T) / np.outer(norms, norms), 0.0)
    np.fill_diagonal(S, 0.0)                         # an entity is not its own neighbour
    return S

def fit_cf(train, ligands, targets, method="item", k=10):
    l = to_idx(train["ligand_key"], ligands)
    t = to_idx(train["target_key"], targets)
    y = train["score"].values
    mu, b_l, b_t = fit_baseline(l, t, y, len(ligands), len(targets))
    R = np.full((len(ligands), len(targets)), np.nan)    # training scores, NaN = not measured
    R[l, t] = y
    E = R - (mu + b_l[:, None] + b_t[None, :])           # deviations from the baseline
    S = None
    if method in ("item", "user"):
        S = centred_cosine(R.T if method == "item" else R)
    return {"method": method, "k": k, "mu": mu, "b_l": b_l, "b_t": b_t,
            "E": E, "S": S, "ligands": ligands, "targets": targets}

def predict(model, ligand_keys, target_keys):
    l = to_idx(ligand_keys, model["ligands"])
    t = to_idx(target_keys, model["targets"])
    base = model["mu"] + model["b_l"][l] + model["b_t"][t]
    method = model["method"]
    if method == "global":
        return np.full(len(l), model["mu"]), np.zeros(len(l), int)
    if method == "ligand_mean":
        return model["mu"] + model["b_l"][l], np.zeros(len(l), int)
    if method == "baseline":
        return base, np.zeros(len(l), int)

    E, S, k = model["E"], model["S"], int(model["k"])
    # item-based: neighbours are targets the ligand has scored; user-based: ligands that scored the target
    query, other = (t, l) if method == "item" else (l, t)
    Eq = E if method == "item" else E.T              # rows indexed by `other`, columns by neighbour entity
    pred, n_used = base.copy(), np.zeros(len(l), int)
    for o in np.unique(other):
        rows = np.where(other == o)[0]
        rated = np.where(~np.isnan(Eq[o]))[0]
        if len(rated) == 0:
            continue
        sims = S[np.ix_(query[rows], rated)]          # (queries for this o) x (rated neighbours)
        sims = np.where(sims > 0, sims, 0.0)
        if len(rated) > k:                            # keep top-k neighbours per query
            cut = np.argpartition(-sims, k - 1, axis=1)[:, :k]
            keep = np.zeros_like(sims, dtype=bool)
            np.put_along_axis(keep, cut, True, axis=1)
            sims = np.where(keep, sims, 0.0)
        wsum = sims.sum(axis=1)
        ok = wsum > 0
        pred[rows[ok]] += (sims[ok] @ Eq[o, rated]) / wsum[ok]
        n_used[rows] = (sims > 0).sum(axis=1)
    return pred, n_used

## Step 3: Metrics

**MSE** between predicted and true scores (lower is better).

**Concordance Index**, via Kendall's $\tau_b$ as suggested in the brief:

$$\mathrm{CI} = \frac{\tau \sqrt{B/A} + 1}{2}$$

$A$ = number of pairs with different true scores, $B$ = number of pairs with different predicted scores. $\mathrm{CI} = 0.5$ is random ordering, $1$ is perfect.

**Ranking (Hit@k):** for each evaluation ligand, its held-out targets in that partition are ranked by predicted score. Hit@$k$ = 1 if the truly best target (highest true score) is in the top $k$. Ligands whose held-out targets all have the same true score (nothing to rank, common at the DAVIS screening limit) are skipped. We also report the mean predicted rank of the true best target and the Hit@$k$ expected from random ranking, because ligands with few held-out targets make Hit@5 easy by chance.

In [ ]:
def n_distinct_pairs(x):
    n = len(x)
    _, counts = np.unique(x, return_counts=True)
    return n * (n - 1) / 2 - (counts * (counts - 1) / 2).sum()

def concordance_index(y, yhat):
    A, B = n_distinct_pairs(y), n_distinct_pairs(yhat)
    if A == 0 or B == 0:
        return 0.5
    tau = kendalltau(y, yhat, variant="b").statistic
    return (tau * np.sqrt(B / A) + 1) / 2

def ranking_metrics(df, ks=(1, 5)):
    rows = []
    for lig, g in df.groupby("ligand_key"):
        if g["score"].nunique() < 2:
            continue
        order = g.sort_values("pred", ascending=False, kind="mergesort")
        best = order["score"] == order["score"].max()
        rank = int(np.argmax(best.values)) + 1          # predicted rank of the (first) true-best target
        n, n_best = len(g), int(best.sum())
        row = {"ligand_key": lig, "n_candidates": n, "rank_of_best": rank}
        for k in ks:
            row[f"hit@{k}"] = float(rank <= k)
            # chance that a random ranking puts at least one of the n_best targets in the top k
            row[f"random hit@{k}"] = 1 - np.prod([(n - n_best - i) / (n - i) for i in range(min(k, n))]) if k < n else 1.0
        rows.append(row)
    return pd.DataFrame(rows)

def evaluate(model, df):
    yhat, n_used = predict(model, df["ligand_key"], df["target_key"])
    scored = df.assign(pred=yhat, n_neighbours=n_used)
    rk = ranking_metrics(scored)
    out = {
        "MSE": np.mean((scored["score"] - scored["pred"]) ** 2),
        "CI": concordance_index(scored["score"].values, scored["pred"].values),
        "Hit@1": rk["hit@1"].mean(), "Hit@5": rk["hit@5"].mean(),
        "random Hit@1": rk["random hit@1"].mean(), "random Hit@5": rk["random hit@5"].mean(),
        "mean rank of best": rk["rank_of_best"].mean(), "ranked ligands": len(rk),
    }
    if model["method"] in ("item", "user"):
        out["fallback %"] = 100 * np.mean(n_used == 0)
    return out, scored

## Step 4: Hyperparameter selection (validation set only)

Grid search over CF direction (item-based / user-based) and neighbourhood size $k$, choosing the lowest validation MSE. The test set is not touched here.

In [ ]:
KS = [2, 3, 5, 10, 20, 40, 80]

best_params, search_logs = {}, {}
for name, df in splits.items():
    train, val = df[df["split"] == "train"], df[df["split"] == "val"]
    ligands, targets = build_index(df)
    rows = []
    for method in ["item", "user"]:
        model = fit_cf(train, ligands, targets, method)      # similarity does not depend on k
        for k in KS:
            model["k"] = k
            rows.append({"method": method, "k": k, "val MSE": evaluate(model, val)[0]["MSE"]})
    log = pd.DataFrame(rows).sort_values("val MSE")
    search_logs[name] = log
    best = log.iloc[0]
    best_params[name] = {"method": best["method"], "k": int(best["k"])}
    print(f"{name.upper()}: best CF = {best_params[name]}, val MSE = {best['val MSE']:.4f}")

pd.concat({n.upper(): log.head(5).reset_index(drop=True) for n, log in search_logs.items()})

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (name, log) in zip(axes, search_logs.items()):
    for method, g in log.groupby("method"):
        g = g.sort_values("k")
        ax.plot(g["k"], g["val MSE"], marker="o", label=f"{method}-based")
    ax.set_xscale("log")
    ax.set_xlabel("Neighbourhood size k")
    ax.set_ylabel("Validation MSE")
    ax.set_title(name.upper())
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR / "cf_hyperparameter_search.png", dpi=200)
plt.show()

## Step 5: Final evaluation

Each model is fitted on training data only with the selected hyperparameters, then evaluated on train, validation and **test (once)**.

The baselines answer *"is CF worth it?"*:
- **global mean**: same score for everything;
- **ligand mean**: the fallback the brief describes, which gives every target of a ligand the same score, so it cannot rank targets at all;
- **baseline** $\mu + b_u + b_i$ (lecture slides 18–22): ranks targets by how strongly they bind on average (a "popularity" ranking).

A pure lookup of known measurements has **no answer** for any validation/test pair, since none of them appear in training.

In [ ]:
final_models, results, scored_test = {}, [], {}
for name, df in splits.items():
    train = df[df["split"] == "train"]
    ligands, targets = build_index(df)
    p = best_params[name]
    final_name = f"CF ({p['method']}-based, k={p['k']})"
    models = {
        "global mean": fit_cf(train, ligands, targets, "global"),
        "ligand mean": fit_cf(train, ligands, targets, "ligand_mean"),
        "baseline (mu + b_u + b_i)": fit_cf(train, ligands, targets, "baseline"),
        final_name: fit_cf(train, ligands, targets, **p),
    }
    final_models[name] = models[final_name]
    for model_name, model in models.items():
        for part in ["train", "val", "test"]:
            metrics, scored = evaluate(model, df[df["split"] == part])
            results.append({"dataset": name.upper(), "model": model_name, "split": part, **metrics})
            if part == "test" and model is final_models[name]:
                scored_test[name] = scored

results_df = pd.DataFrame(results)
results_df[results_df["split"] == "test"].set_index(["dataset", "model"]).drop(columns="split").round(4)

In [ ]:
# Train / validation / test for the final CF models (checks for over-fitting)
cf_rows = results_df[results_df["model"].str.startswith("CF")]
cf_rows.pivot_table(index=["dataset", "model"], columns="split", values=["MSE", "CI"]).round(4)

### Where does CF help, and where does it fall back?

DAVIS: error on screening-limit pairs ($\mathrm{p}K_d = 5$, censored "weaker than measurable") vs. measured binders. KIBA: how many neighbours each test prediction could use, and whether ligands with few training observations get worse predictions.

In [ ]:
d = scored_test["davis"]
at_limit = np.isclose(d["score"], SCALES["davis"]["screening_limit"])
base_davis = fit_cf(splits["davis"][splits["davis"]["split"] == "train"], *build_index(splits["davis"]), "baseline")
d_base, _ = predict(base_davis, d["ligand_key"], d["target_key"])
davis_breakdown = pd.DataFrame({
    "test pairs": [at_limit.sum(), (~at_limit).sum()],
    "CF MSE": [np.mean((d["score"] - d["pred"])[at_limit] ** 2), np.mean((d["score"] - d["pred"])[~at_limit] ** 2)],
    "baseline MSE": [np.mean((d["score"] - d_base)[at_limit] ** 2), np.mean((d["score"] - d_base)[~at_limit] ** 2)],
}, index=["at screening limit (pKd = 5)", "measured binding (pKd > 5)"]).round(4)
display(davis_breakdown)

k_df = scored_test["kiba"]
kiba_train = splits["kiba"][splits["kiba"]["split"] == "train"]
n_train_obs = kiba_train.groupby("ligand_key").size()
k_df = k_df.assign(ligand_train_obs=k_df["ligand_key"].map(n_train_obs))
k_df["ligand training observations"] = pd.cut(k_df["ligand_train_obs"], [0, 10, 20, 50, 100, 1000])
kiba_breakdown = k_df.groupby("ligand training observations", observed=True).agg(
    test_pairs=("score", "size"),
    CF_MSE=("pred", lambda s: np.mean((k_df.loc[s.index, "score"] - s) ** 2)),
    mean_neighbours=("n_neighbours", "mean"),
    fallback_pct=("n_neighbours", lambda s: 100 * np.mean(s == 0)),
).round(3)
kiba_breakdown

## Step 6: Save the models and verify reloading

Each checkpoint stores the full CF state (ligand/target vocabularies, $\mu$, biases, training deviation matrix, similarity matrix, hyperparameters) and the score-scale information. Reloading must reproduce the predictions exactly.

In [ ]:
def save_model(model, path, scale):
    np.savez_compressed(
        path,
        method=model["method"], k=model["k"],
        mu=model["mu"], b_l=model["b_l"], b_t=model["b_t"], E=model["E"], S=model["S"],
        ligands=model["ligands"], targets=model["targets"],
        scale=json.dumps(scale), seed=SEED,
    )

def load_model(path):
    z = np.load(path, allow_pickle=False)
    model = {key: z[key] for key in ["mu", "b_l", "b_t", "E", "S", "ligands", "targets"]}
    model.update(method=str(z["method"]), k=int(z["k"]), mu=float(z["mu"]), scale=json.loads(str(z["scale"])))
    return model

for name, model in final_models.items():
    path = MODEL_DIR / f"{name}_cf.npz"
    save_model(model, path, SCALES[name])
    reloaded = load_model(path)
    test = splits[name][splits[name]["split"] == "test"]
    before, _ = predict(model, test["ligand_key"], test["target_key"])
    after, _ = predict(reloaded, test["ligand_key"], test["target_key"])
    print(f"{path}: {path.stat().st_size / 1e6:.1f} MB, scale = {reloaded['scale']['score_name']}, "
          f"identical predictions after reload: {np.array_equal(before, after)}")

## Step 7: Case study (predictions from the reloaded models)

Three fixed test ligands per dataset (the IDs suggested in the brief). For each one, all of its **test** targets are ranked by the model loaded from disk. The truly best test target is labelled **correct**, the others **wrong**.

In [ ]:
CASES = {"davis": ["10127622", "11338033", "44259"],
         "kiba": ["CHEMBL1967538", "CHEMBL2004544", "CHEMBL565612"]}

loaded = {name: load_model(MODEL_DIR / f"{name}_cf.npz") for name in CASES}
case_summary = []
for name, ligs in CASES.items():
    df = splits[name]
    test = df[df["split"] == "test"]
    model = loaded[name]
    for lig in ligs:
        g = test[test["ligand_key"] == lig]
        n_train = int(((df["split"] == "train") & (df["ligand_key"] == lig)).sum())
        if g.empty:
            print(f"{name.upper()} {lig}: no test targets in this split, choose another ligand")
            continue
        pred, n_nb = predict(model, g["ligand_key"], g["target_key"])
        g = g.assign(pred=pred, neighbours=n_nb).sort_values("pred", ascending=False, kind="mergesort")
        g["predicted rank"] = np.arange(1, len(g) + 1)
        best = g["score"] == g["score"].max()
        g["label"] = np.where(best, "correct", "wrong")
        rank_best = int(g.loc[best, "predicted rank"].min())
        case_summary.append({
            "dataset": name.upper(), "ligand": lig, "train obs": n_train, "test targets": len(g),
            "true best score": g["score"].max(), "tied best targets": int(best.sum()),
            "MSE": np.mean((g["score"] - g["pred"]) ** 2),
            "CI": concordance_index(g["score"].values, g["pred"].values),
            "rank of true best": rank_best, "Hit@1": rank_best <= 1, "Hit@5": rank_best <= 5,
        })
        print(f"\n{name.upper()} ligand {lig}  ({n_train} training scores, {len(g)} test targets)")
        display(g[["predicted rank", "target_key", "pred", "score", "label", "neighbours"]]
                .rename(columns={"target_key": "target", "pred": "predicted", "score": "true"})
                .head(10).round(3).reset_index(drop=True))

pd.DataFrame(case_summary).round(3)